# Imports

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas
import unicodedata

# Constants

In [ ]:
base_url = 'https://mibici.net'

# Fetch website

In [ ]:
source_url = f'{base_url}/es/datos-abiertos/'

headers = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36'
}

In [ ]:
r = requests.get(source_url, headers=headers)

r

In [ ]:
r.text

# Parse HTML for file links

In [ ]:
soup = BeautifulSoup(r.text, 'html.parser')

In [ ]:
links = soup.find_all('a')

In [ ]:
parent_dir = '/site/assets/files'

file_links = []
file_paths = []

for link in links:
    link_href = link.get('href')
    if not link_href.startswith(parent_dir): continue
    if not 'datos_abiertos' in link_href: continue
    file_links.append(link)
    file_paths.append(link_href)

# file_links = [link for link in links if link.get('href').startswith(parent_dir) and ('datos_abiertos' in link.get('href'))]
# links[10].get('href')

# Filter by date range

In [ ]:
df_paths = pandas.DataFrame(file_paths, columns=['path'])

df_paths.head()

In [ ]:
df_date = ( df_paths.path
    .str.extract(r'(\d{4}_\d{2})')
    .rename(columns={0:'file_date'})
)

df_date.head()

In [ ]:
df_paths['file_date'] = pandas.to_datetime(df_date['file_date'], format='%Y_%m')

df_paths.head()

In [ ]:
date_start = '2016-04-01'
date_end = '2016-10-01'
df_paths.query('(file_date >= @date_start) and (file_date <= @date_end)')

In [ ]:
df_paths.path.to_list()

# Download a file

In [ ]:
filepath = df_paths.loc[0, 'path']

file_url = base_url + filepath

file_url

In [ ]:
r_file = requests.get(file_url, headers=headers)

r_file

In [ ]:
type(r_file.content)

In [ ]:
filename = filepath.split('/')[-1]
print(filename)

with open(filename, 'wb') as f:
    f.write(r_file.content)

# Read file

In [ ]:
df = pandas.read_csv(filename)

df

In [ ]:
cols = list(df.columns)
cols

In [ ]:
for col in cols:
    col_lower = col.lower()
    col_norm = unicodedata.normalize('NFKD', col_lower)
    col_final = col_norm.encode('ascii', 'ignore').decode('ascii')

    print(col, '\n', col_norm, '\n', col_final)
    print('')